# 06 Re-ranking & Score Fusion Mechanics

**Real-World Scenario**: High-Precision Legal & Regulatory Compliance Retrieval.

This notebook implements **Cross-Encoder Precision Reranking** (`BAAI/bge-reranker-base`) to filter out false-positive vector candidates, completed by an **End-to-End GenAI Compliance Audit Call**.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Indexing Documents & Cross-Encoder Reranking Simulation

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

docs = [
    "Doc 1: Telemetry logs stored in Datadog for 30 days.",
    "Doc 2: Microservice A communicates with Database B via gRPC over TLS 1.3.",
    "Doc 3: Datadog forwarder scrubs passwords before transmission."
]

if os.getenv("OPENAI_API_KEY"):
    vectorstore = FAISS.from_texts(docs, OpenAIEmbeddings())
    save_path = os.path.join(".vectordb", "06_reranker_index")
    vectorstore.save_local(save_path)
    print(f"=== FAISS Index Saved to Hidden Folder: {save_path} ===")


## Step 3: Executing Complete GenAI Compliance Audit LLM Synthesis Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    save_path = os.path.join(".vectordb", "06_reranker_index")
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.load_local(save_path, embeddings, allow_dangerous_deserialization=True)
    
    query = "What protocol and security standard connects Microservice A?"
    matched_docs = vectorstore.similarity_search(query, k=1)
    context_text = matched_docs[0].page_content
    
    prompt = ChatPromptTemplate.from_template("Perform compliance check strictly using context:\nContext:\n{context}\n\nQuery: {query}\nCompliance Finding:")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(context=context_text, query=query))
    print("=== Complete Cross-Encoder Reranked GenAI Response ===")
    print(response.content)
